# Semana 4 — Interferencia Cuántica y Fases
## El mecanismo que hace útil la computación cuántica

**Objetivo:** Entender cómo la interferencia constructiva amplifica respuestas correctas y la destructiva cancela las incorrectas. Este es el motor de casi todos los algoritmos cuánticos.

**Hito verificable:** Puedes construir un circuito simple que use interferencia para distinguir funciones constantes de funciones balanceadas (precursor del algoritmo de Deutsch).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('Librerías importadas correctamente')

## Ejercicio 4.1 — Fase global vs. fase relativa
La fase global es inobservable. La fase relativa entre |0⟩ y |1⟩ sí importa.

In [ ]:
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Fase global: e^(iφ)|ψ⟩ — mismas probabilidades en CUALQUIER base
psi_original = np.array([1, 1], dtype=complex) / np.sqrt(2)  # |+⟩
psi_fase_global = np.exp(1j * np.pi/3) * psi_original

print('=== Fase global (inobservable) ===')
print(f'|+⟩ original:   probs = {np.round(np.abs(psi_original)**2, 4)}')
print(f'e^(iπ/3)|+⟩:    probs = {np.round(np.abs(psi_fase_global)**2, 4)}')
print(f'Tras H:  original = {np.round(np.abs(H @ psi_original)**2, 4)}')
print(f'Tras H:  con fase = {np.round(np.abs(H @ psi_fase_global)**2, 4)}')
print('→ La fase global no produce ningún efecto observable.')
print()

# Fase relativa: |0⟩ + e^(iφ)|1⟩ — mismas probs en base Z, DISTINTAS en base X
phi = np.pi/2  # fase π/2 → el estado |+i⟩
psi_sin_fase = np.array([1, 1], dtype=complex) / np.sqrt(2)          # |+⟩
psi_con_fase = np.array([1, np.exp(1j*phi)], dtype=complex) / np.sqrt(2)  # |+i⟩

print('=== Fase relativa (observable) ===')
print(f'|+⟩   probs base Z:  {np.round(np.abs(psi_sin_fase)**2, 4)}')
print(f'|+i⟩  probs base Z:  {np.round(np.abs(psi_con_fase)**2, 4)}  (iguales)')
print(f'|+⟩   probs base X (tras H):  {np.round(np.abs(H @ psi_sin_fase)**2, 4)}')
print(f'|+i⟩  probs base X (tras H):  {np.round(np.abs(H @ psi_con_fase)**2, 4)}  (¡diferentes!)')
print('→ La fase relativa es detectable midiendo en una base diferente.')

## Ejercicio 4.2 — Interferencia constructiva y destructiva
Las amplitudes se suman (interferencia constructiva) o se cancelan (destructiva).

In [ ]:
# Circuito: H → Z → H
# H crea superposición, Z aplica fase -1 a |1⟩, H interfiere
psi_0 = ket_0.copy()

paso1 = H @ psi_0        # H|0⟩ = |+⟩
paso2 = Z @ paso1        # Z|+⟩ = |−⟩
paso3 = H @ paso2        # H|−⟩ = |1⟩

print('Circuito H → Z → H aplicado a |0⟩:')
print(f'  Después de H:  {np.round(paso1, 4)}')
print(f'  Después de Z:  {np.round(paso2, 4)}')
print(f'  Después de H:  {np.round(paso3, 4)}')
print(f'  Probs finales: {np.round(np.abs(paso3)**2, 4)}')
print()
print('Análisis de amplitudes:')
print('  H|0⟩ = (|0⟩+|1⟩)/√2  →  amplitud ½ para cada rama')
print('  Z introduce -1 en |1⟩ →  amplitudes (½, -½)')
print('  H suma las ramas:')
print('    amplitud de |0⟩ = (½ + (-½))/√2 = 0     → DESTRUCTIVA')
print('    amplitud de |1⟩ = (½ - (-½))/√2 = 1/√2·√2 = 1 → CONSTRUCTIVA')
print('  Resultado: |1⟩ con certeza')

## Ejercicio 4.3 — El algoritmo de Deutsch
Primer algoritmo cuántico con ventaja real. Determina si f:{0,1}→{0,1} es constante o balanceada con UNA sola consulta.

In [ ]:
I_mat = np.eye(4, dtype=complex)  # identidad 4×4
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)

def Uf_constante_0(n=2):
    """Oráculo f(x)=0 (constante): no hace nada."""
    return np.eye(4, dtype=complex)

def Uf_constante_1(n=2):
    """Oráculo f(x)=1 (constante): flipa el qubit auxiliar."""
    return np.kron(np.eye(2, dtype=complex), np.array([[0,1],[1,0]], dtype=complex))

def Uf_identidad(n=2):
    """Oráculo f(x)=x (balanceada): CNOT."""
    return CNOT

def Uf_negacion(n=2):
    """Oráculo f(x)=NOT x (balanceada): CNOT luego X en primer qubit."""
    X_mat = np.array([[0,1],[1,0]], dtype=complex)
    return CNOT @ np.kron(X_mat, np.eye(2, dtype=complex))

def deutsch(Uf):
    """Circuito de Deutsch. Devuelve 0 si constante, 1 si balanceada."""
    # Estado inicial: |0⟩|1⟩
    ket_0 = np.array([1, 0], dtype=complex)
    ket_1 = np.array([0, 1], dtype=complex)
    psi = np.kron(ket_0, ket_1)
    
    # H⊗H
    H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
    HH = np.kron(H, H)
    psi = HH @ psi
    
    # Oráculo
    psi = Uf @ psi
    
    # H⊗I en el primer qubit
    HI = np.kron(H, np.eye(2, dtype=complex))
    psi = HI @ psi
    
    # Medir primer qubit: P(0) vs P(1)
    p0 = abs(psi[0])**2 + abs(psi[1])**2  # estados |00⟩ y |01⟩
    p1 = abs(psi[2])**2 + abs(psi[3])**2  # estados |10⟩ y |11⟩
    
    return 0 if p0 > 0.5 else 1, p0, p1

funciones = [
    ('f(x)=0 (constante)',   Uf_constante_0),
    ('f(x)=1 (constante)',   Uf_constante_1),
    ('f(x)=x (balanceada)',  Uf_identidad),
    ('f(x)=¬x (balanceada)', Uf_negacion),
]

print('=== Algoritmo de Deutsch ===')
print(f'{"Función":25}  {"Resultado":10}  {"P(|0⟩)":8}  {"P(|1⟩)":8}')
print('-' * 60)
for nombre, Uf in funciones:
    resultado, p0, p1 = deutsch(Uf)
    tipo = 'constante' if resultado == 0 else 'balanceada'
    print(f'{nombre:25}  {tipo:10}  {p0:.4f}    {p1:.4f}')

print()
print('→ 1 consulta cuántica vs 2 consultas clásicas. Ventaja cuántica.')

## Ejercicio 4.4 — Visualizar la interferencia
Graficar cómo cambia la probabilidad de medir |0⟩ en función de la fase relativa φ.

In [ ]:
# Estado: (|0⟩ + e^(iφ)|1⟩)/√2, luego medimos en la base X (aplicamos H)
phis = np.linspace(0, 2*np.pi, 500)
p0_list = []

for phi in phis:
    psi = np.array([1, np.exp(1j*phi)], dtype=complex) / np.sqrt(2)
    psi_medido = H @ psi  # cambio a base X
    p0_list.append(abs(psi_medido[0])**2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(phis / np.pi, p0_list, 'b-', linewidth=2, label='P(|0⟩) tras H')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='50%')
ax.set_xlabel('Fase relativa φ (en unidades de π)')
ax.set_ylabel('P(medir |0⟩)')
ax.set_title('Interferencia: P(|0⟩) en función de la fase relativa φ')
ax.legend()
ax.set_xticks([0, 0.5, 1, 1.5, 2])
ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
plt.tight_layout()
plt.savefig('interferencia_fase.png', dpi=100, bbox_inches='tight')
plt.show()
print('φ=0 → interferencia constructiva para |0⟩ (P=1)')
print('φ=π → interferencia destructiva para |0⟩ (P=0)')

## Mini reto — Completar antes de avanzar a la Semana 5

Implementa el algoritmo de Deutsch-Jozsa para n=3 qubits. El problema: determinar si f:{0,1}³→{0,1} es constante o balanceada con UNA sola consulta cuántica (se necesitan hasta 2^(n-1)+1 = 5 consultas clásicas en el peor caso).

Crea al menos 3 oráculos: uno constante y dos balanceados.

In [ ]:
# TU CÓDIGO AQUÍ
def deutsch_jozsa_3qubits(Uf: np.ndarray) -> str:
    """Devuelve 'constante' o 'balanceada'."""
    # TODO: implementar
    # Pista: el estado inicial es |000⟩|1⟩ (3 qubits de consulta + 1 auxiliar)
    pass

## ✅ Hito de la Semana 4

Antes de avanzar, comprueba que puedes:

- [ ] Distinguir fase global (inobservable) de fase relativa (observable)
- [ ] Construir ejemplos de interferencia constructiva y destructiva
- [ ] Implementar el algoritmo de Deutsch y verificar los 4 casos
- [ ] Explicar por qué el circuito H→Uf→H usa interferencia
- [ ] Graficar P(|0⟩) en función de la fase relativa

## Reflexión (escribe aquí tu respuesta)

**¿Por qué el algoritmo de Deutsch necesita exactamente una consulta al oráculo?**

*(tu respuesta aquí)*

**¿Qué rol juega la interferencia en la ventaja cuántica?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 5 — Sistemas Multi-Qubit y Entrelazamiento*